In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
import glob
import sys
import copy
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor as XGBR

## XGB

In [2]:
data_path = r"path\Train_Test_Verification_Data.xlsx"

train_data = pd.read_excel(data_path, sheet_name="Train")
test_data = pd.read_excel(data_path, sheet_name="Test")


feature_train = train_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_train = train_data.iloc[:, -1].to_numpy(dtype=np.float32)

feature_test = test_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_test = test_data.iloc[:, -1].to_numpy(dtype=np.float32)

print("Train:", feature_train.shape, label_train.shape)
print("Test :", feature_test.shape, label_test.shape)

Train: (71520, 10) (71520,)
Test : (30652, 10) (30652,)


In [4]:
xgb = XGBR(n_estimators=375,max_depth =3,random_state = 300).fit(feature_train,label_train)
xgb.score(feature_test,label_test)

0.9858113762337618

## MQR

In [2]:
import pandas as pd
import numpy as np


data_path = r"path\Train_Test_Verification_Data.xlsx"

train_data = pd.read_excel(data_path, sheet_name="Train")
test_data = pd.read_excel(data_path, sheet_name="Test")


single_bc_path = r"path\Train_Test_Verification_Data.xlsx"
single_bc_data = pd.read_excel(data_path,sheet_name="1BC")

print("Single BC data shape:", single_bc_data.shape)
print("Single BC columns:")
print(single_bc_data.columns.tolist())

display(single_bc_data.head())


bc_cols = [
    "Cellulose",
    "Hemicellulose",
    "Lignin",
    "PE",
    "PS",
    "PP",
    "PET",
    "PVC",
    "Starch"
]

time_col = "Time"
label_col = "TG"

bc_name_map = {
    "Cellulose": "Cellulose",
    "Hemicellulose": "Hemicellulose",
    "Lignin": "Lignin",
    "PE": "PE",
    "PS": "PS",
    "PP": "PP",
    "PET": "PET",
    "PVC": "PVC",
    "Starch": "Starch"
}



def build_single_bc_tg_matrix(single_bc_data, bc_cols, bc_name_map):
    single_tg_dict = {}

    for bc in bc_cols:
        single_col = bc_name_map[bc]
        one_bc_curve = single_bc_data.loc[
            single_bc_data[single_col] == 1,
            ["Time", "TG(wt.%)"]
        ].copy()

        one_bc_curve = one_bc_curve.reset_index(drop=True)

        single_tg_dict[bc] = one_bc_curve["TG(wt.%)"].to_numpy(dtype=np.float32)

    first_bc = bc_cols[0]
    first_single_col = bc_name_map[first_bc]

    time_array = single_bc_data.loc[
        single_bc_data[first_single_col] == 1,
        "Time"
    ].reset_index(drop=True).to_numpy(dtype=np.float32)

    single_tg_df = pd.DataFrame(single_tg_dict)
    single_tg_df.insert(0, time_col, time_array)

    return single_tg_df


single_tg_df = build_single_bc_tg_matrix(
    single_bc_data=single_bc_data,
    bc_cols=bc_cols,
    bc_name_map=bc_name_map
)



def build_mqr_features_old_fusion(
    df,
    single_tg_df,
    bc_cols,
    time_col,
    points_per_experiment=574,
    use_percent_composition=True
):
   

    df = df.copy().reset_index(drop=True)

    bcs_values = df[bc_cols].to_numpy(dtype=np.float32)

    max_bcs = np.nanmax(bcs_values)

    if use_percent_composition and max_bcs <= 1.5:
        bcs_values = bcs_values * 100.0

    data_time = df[time_col].to_numpy(dtype=np.float32)
    single_time = single_tg_df[time_col].to_numpy(dtype=np.float32)

    time_indices = np.array([
        np.argmin(np.abs(single_time - t))
        for t in data_time
    ])

    max_time_diff = np.max(np.abs(data_time - single_time[time_indices]))


    single_tg_values = single_tg_df[bc_cols].to_numpy(dtype=np.float32)
    single_tg_for_each_row = single_tg_values[time_indices, :]

    #Features mxiture
    fused_features = bcs_values * single_tg_for_each_row / 100.0

    fused_df = pd.DataFrame(
        fused_features,
        columns=bc_cols,
        index=df.index
    )

    return fused_df



X_train_df = build_mqr_features_old_fusion(
    df=train_data,
    single_tg_df=single_tg_df,
    bc_cols=bc_cols,
    time_col=time_col,
    points_per_experiment=574,
    use_percent_composition=True
)

X_test_df = build_mqr_features_old_fusion(
    df=test_data,
    single_tg_df=single_tg_df,
    bc_cols=bc_cols,
    time_col=time_col,
    points_per_experiment=574,
    use_percent_composition=True
)

y_train = train_data[label_col].to_numpy(dtype=np.float32).ravel()
y_test = test_data[label_col].to_numpy(dtype=np.float32).ravel()

X_train = X_train_df.copy()
X_test = X_test_df.copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

display(X_train_df.head())
display(X_test_df.head())

Single BC data shape: (5166, 11)
Single BC columns:
['Cellulose', 'Hemicellulose', 'Lignin', 'PE', 'PS', 'PP', 'PET', 'PVC', 'Starch', 'Time', 'TG(wt.%)']


,Cellulose,Hemicellulose,Lignin,PE,PS,PP,PET,PVC,Starch,Time,TG(wt.%)
0,1,0,0,0,0,0,0,0,0,0.00000,100.0
1,1,0,0,0,0,0,0,0,0,0.17473,100.0
2,1,0,0,0,0,0,0,0,0,0.34946,100.0
3,1,0,0,0,0,0,0,0,0,0.52419,100.0
4,1,0,0,0,0,0,0,0,0,0.69892,100.0


X_train shape: (71520, 9)
X_test shape: (30652, 9)
y_train shape: (71520,)
y_test shape: (30652,)


,Cellulose,Hemicellulose,Lignin,PE,PS,PP,PET,PVC,Starch
0,13.62709,0.000000,0.000000,0.000000,0.000000,19.336287,0.000000,0.000000,0.0
1,0.00000,0.000000,33.264023,0.000000,33.326996,0.000000,0.000000,33.240749,0.0
2,0.00000,0.000000,0.000000,-0.281181,0.006681,-0.000000,0.000000,0.000000,0.0
3,0.00000,6.080619,0.000000,0.000000,0.000000,0.000000,6.566424,0.000000,0.0
4,0.00000,0.000000,74.334885,0.000000,0.000000,19.915380,0.000000,0.000000,0.0


,Cellulose,Hemicellulose,Lignin,PE,PS,PP,PET,PVC,Starch
0,0.000000,31.893150,0.000000,0.000000,0.0,0.0,5.872550,0.000000,0.0
1,0.000000,13.341883,0.000000,-0.000000,0.0,-0.0,2.976357,0.000000,0.0
2,1.929157,3.444613,12.449838,-0.000000,0.0,-0.0,3.713798,0.000000,0.0
3,2.821437,0.000000,0.000000,-0.000000,0.0,-0.0,0.000000,3.798095,0.0
4,33.218765,0.000000,0.000000,33.325329,0.0,0.0,0.000000,33.232922,0.0


In [3]:
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
feature_names = poly.get_feature_names_out(bc_cols)

In [4]:
model = LinearRegression()
model.fit(X_train_poly, y_train)
feature_names = poly.get_feature_names_out(bc_cols)

In [5]:
y_pred = model.predict(X_train_poly)
r2_train = r2_score(y_train, y_pred)
mean_squared_error(y_train, y_pred,squared=False)

7.8497434

In [6]:
r2_train

0.9535986099991313

In [7]:
y_pred = model.predict(X_test_poly)
r2_2 = r2_score(y_test, y_pred)
mean_squared_error(y_test, y_pred,squared=False)

6.968463

In [8]:
r2_2

0.9585817990395221